In [3]:
import torch

# Test Linear

In [1]:
from modules import LinearLayer

linear1 = LinearLayer(in_features=4, out_features=3)
linear1.W.shape

torch.Size([3, 4])

In [4]:
x1d = torch.randn(4)           # (d_in)
x2d = torch.randn(5, 4)        # (batch, d_in)
x3d = torch.randn(2, 5, 4)     # (batch, seq, d_in)

print(linear1(x1d).shape)
print(linear1(x2d).shape)
print(linear1(x3d).shape)

torch.Size([3])
torch.Size([5, 3])
torch.Size([2, 5, 3])


# Test Embedding

In [5]:
from modules import EmbeddingLayer

emb = EmbeddingLayer(10, 4)  # vocab=10, d_model=4

x0 = torch.tensor(3)                     # scalar
x1 = torch.tensor([1, 2, 3])              # (seq_len,)
x2 = torch.tensor([[1, 2], [3, 4]])       # (batch, seq_len)

print(emb(x0).shape, emb(x0).grad_fn)
print(emb(x1).shape, emb(x1).grad_fn)
print(emb(x2).shape, emb(x2).grad_fn)

torch.Size([4]) <SelectBackward0 object at 0x10842afe0>
torch.Size([3, 4]) <IndexBackward0 object at 0x12fe25960>
torch.Size([2, 2, 4]) <IndexBackward0 object at 0x10842afe0>


In [6]:
emb(x2).grad_fn

# Test RMSNorm

In [7]:
from modules import RMSNormLayer

norm = RMSNormLayer(4)
x = torch.randn(2, 3, 4)  # (batch, seq_len, d_model)
out = norm(x)

print(out.shape)  # should be (2, 3, 4) — same as input
print(out.dtype)  # should match x.dtype
print(x.dtype)

torch.Size([2, 3, 4])
torch.float32
torch.float32


# Test SwiGLU

In [8]:
from modules import SwiGLU

swiglu = SwiGLU(d_model=8, d_ff=24)
x = torch.randn(2, 5, 8)
out = swiglu(x)

print(out.shape)  # what do you expect, and what do you get?

torch.Size([2, 5, 8])


In [9]:
buf = torch.randn(10, 4)  # (max_seq_len, d_k//2)
pos = torch.tensor([0, 1, 2])  # (seq_len,)
print(buf[pos].shape)  # what do you get?

pos_batched = torch.tensor([[0,1,2],[3,4,5]])  # (batch, seq_len)
print(buf[pos_batched].shape)  # what do you get?

torch.Size([3, 4])
torch.Size([2, 3, 4])


# Test RoPE

In [10]:
from modules import RoPE

# Test Softmax

In [37]:
x = torch.randn(size=(2, 2, 4))
x

tensor([[[-0.7540,  0.3002, -0.7822,  1.8178],
         [-0.6988,  0.0184,  0.2096,  0.3922]],

        [[-1.0572,  0.7123,  0.2382, -0.1119],
         [ 0.1268, -0.4889, -1.7055, -1.5716]]])

In [39]:
max_value = torch.max(x, dim=2, keepdim=True).values
max_value

tensor([[[1.8178],
         [0.3922]],

        [[0.7123],
         [0.1268]]])

In [40]:
x-max_value

tensor([[[-2.5719, -1.5177, -2.6001,  0.0000],
         [-1.0910, -0.3738, -0.1826,  0.0000]],

        [[-1.7696,  0.0000, -0.4741, -0.8242],
         [ 0.0000, -0.6157, -1.8323, -1.6984]]])

In [41]:
exp = torch.exp(x - max_value)
exp

tensor([[[0.0764, 0.2192, 0.0743, 1.0000],
         [0.3359, 0.6881, 0.8331, 1.0000]],

        [[0.1704, 1.0000, 0.6224, 0.4386],
         [1.0000, 0.5403, 0.1600, 0.1830]]])

In [48]:
softmax_val = torch.exp(exp)/torch.sum(torch.exp(exp), dim=2, keepdim=True)
softmax_val

tensor([[[0.1764, 0.2035, 0.1760, 0.4442],
         [0.1664, 0.2367, 0.2736, 0.3233]],

        [[0.1620, 0.3715, 0.2546, 0.2119],
         [0.3992, 0.2521, 0.1724, 0.1763]]])

In [50]:
torch.sum(softmax_val, dim=2)

tensor([[1.0000, 1.0000],
        [1.0000, 1.0000]])